# 《LangChain OutputParser输出解析器实验》

## 一、实验目的
1. 理解LangChain输出解析器的作用与分类
2. 掌握StrOutputParser字符串解析器的使用方法
3. 掌握JsonOutputParser解析器配合Pydantic的结构化输出
4. 掌握CommaSeparatedListOutputParser列表解析器的使用
5. 掌握XMLOutputParser解析器的使用方法
6. 掌握PydanticOutputParser解析器的强类型约束输出


## 二、实验环境
- 系统：Windows 10
- Python版本：3.10
- 虚拟环境：Miniconda
- 开发工具：VS Code / Jupyter Notebook
- 依赖库：langchain、langchain-core、langchain-deepseek、pydantic、python-dotenv
- 模型：DeepSeek（deepseek-chat）


## 三、实验原理

### 3.1 StrOutputParser字符串解析器

StrOutputParser是LangChain中最简单的输出解析器，它可以简单地将任何输入转换为字符串。从结果中提取content字段转换为字符串输出。

### 3.2 JsonOutputParser解析器

JsonOutputParser，即JSON输出解析器，是一种用于将大模型的自由文本输出转换为结构化JSON数据的工具。适合需要严格结构化输出的场景，比如API调用、数据存储或下游任务处理。

### 3.3 CommaSeparatedListOutputParser列表解析器

利用CommaSeparatedListOutputParser解析器，可以将模型的文本响应转换为一个用逗号分隔的列表（List[str]）。

### 3.4 XMLOutputParser解析器

XMLOutputParser将模型的自由文本输出转换为可编程处理的XML数据。注意：它不会直接将模型的输出保持为原始XML字符串，而是会解析XML并转换成Python字典（或类似结构化的数据），目的是为了方便程序后续处理数据。

### 3.5 PydanticOutputParser解析器

PydanticOutputParser是LangChain输出解析器体系中最常用、最强大的结构化解析器之一。它与JsonOutputParser类似，但功能更强——能直接基于Pydantic模型定义输出结构，并利用其类型校验与自动文档能力。对于结构更复杂、具有强类型约束的需求，PydanticOutputParser则是最佳选择。

## 四、实验内容

### 4.1 StrOutputParser字符串解析器

StrOutputParser是LangChain中最简单的输出解析器，它可以简单地将任何输入转换为字符串。从结果中提取content字段转换为字符串输出。

In [1]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

deepseek_api_key = os.getenv("DEEPSEEK_API_KEY")

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_deepseek import ChatDeepSeek
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

# 创建聊天提示模板
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}，请简短回答我提出的问题"),
    ("human", "请回答:{question}")
])

# 使用指定的角色和问题生成具体的提示内容
prompt = chat_prompt.invoke({"role": "AI助手", "question": "什么是LangChain"})
logger.info(prompt)

# 初始化Deepseek聊天模型
model = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=deepseek_api_key,)

# 调用模型获取回答结果
result = model.invoke(prompt)
logger.info(f"模型原始输出:\n{result}")

# 创建字符串输出解析器，用于解析模型返回的结果
parser = StrOutputParser()

# 打印解析后的结构化结果
response = parser.invoke(result)
logger.info(f"解析后的结构化结果:\n{response}")

# 打印类型
logger.info(f"结果类型: {type(response)}")

INFO:root:messages=[SystemMessage(content='你是一个AI助手，请简短回答我提出的问题', additional_kwargs={}, response_metadata={}), HumanMessage(content='请回答:什么是LangChain', additional_kwargs={}, response_metadata={})]
INFO:httpx:HTTP Request: POST https://api.deepseek.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:root:模型原始输出:
content='LangChain是一个用于构建基于大语言模型（LLM）应用的框架。它提供模块化工具，帮助开发者将LLM与外部数据、API等连接，简化复杂工作流的开发。' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 21, 'total_tokens': 62, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 21}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '741c4ca6-69c2-4868-a926-6289f4c8b70d', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e5e81-89a4-77a1-a8a7-dfd4c4599049-0' tool_calls=[] invalid_tool

### 4.2 JsonOutputParser解析器

JsonOutputParser，即JSON输出解析器，是一种用于将大模型的自由文本输出转换为结构化JSON数据的工具。适合需要严格结构化输出的场景，比如API调用、数据存储或下游任务处理。

#### 通过提示词指定返回JSON格式

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_deepseek import ChatDeepSeek
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

# 创建聊天提示模板，在系统消息中要求返回json格式
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个{role}，请简短回答我提出的问题，结果返回json格式，q字段表示问题，a字段表示答案。"),
    ("human", "请回答:{question}")
])

prompt = chat_prompt.invoke({"role": "AI助手", "question": "什么是LangChain"})
logger.info(prompt)

model = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=deepseek_api_key)

result = model.invoke(prompt)
logger.info(f"模型原始输出:\n{result}")

# 创建JSON输出解析器实例
parser = JsonOutputParser()
response = parser.invoke(result)
logger.info(f"解析后的结构化结果:\n{response}")
logger.info(f"结果类型: {type(response)}")

#### 使用get_format_instructions自动生成格式说明

get_format_instructions会根据你定义的Pydantic数据规则自动生成一段格式说明文字，发给AI后，AI就会严格按照你要求的格式回答，不会乱输出。

In [ ]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_deepseek import ChatDeepSeek
from pydantic import BaseModel, Field
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

class Person(BaseModel):
    time: str = Field(description="时间")
    person: str = Field(description="人物")
    event: str = Field(description="事件")

# 创建JSON输出解析器，绑定Person模型
parser = JsonOutputParser(pydantic_object=Person)

# 获取格式化指令
format_instructions = parser.get_format_instructions()
print("自动生成的格式指令：\n", format_instructions)

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手，你只能输出结构化JSON数据。"),
    ("human", "请生成一个关于{topic}的新闻。{format_instructions}")
])

prompt = chat_prompt.format_messages(topic="小米", format_instructions=format_instructions)
logger.info(prompt)

model = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=deepseek_api_key)

result = model.invoke(prompt)
logger.info(f"模型原始输出:\n{result}")

response = parser.invoke(result)
logger.info(f"解析后的结构化结果:\n{response}")
logger.info(f"结果类型: {type(response)}")

### 4.3 CommaSeparatedListOutputParser列表解析器

利用CommaSeparatedListOutputParser解析器，可以将模型的文本响应转换为一个用逗号分隔的列表（List[str]）。

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_deepseek import ChatDeepSeek
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

# 创建逗号分隔列表输出解析器实例
parser = CommaSeparatedListOutputParser()

# 获取格式化指令
format_instructions = parser.get_format_instructions()

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", f"你是一个AI助手，你只能输出结构化列表数据。{format_instructions}"),
    ("human", "请生成5个关于{topic}的内容")
])

prompt = chat_prompt.format_messages(topic="小米", format_instructions=format_instructions)
logger.info(prompt)

model = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=deepseek_api_key)

result = model.invoke(prompt)
logger.info(f"模型原始输出:\n{result}")

# 使用解析器将模型返回的结果转换为结构化列表
response = parser.invoke(result)
logger.info(f"解析后的结构化结果:\n{response}")
logger.info(f"结果类型: {type(response)}")

### 4.4 XMLOutputParser解析器

XMLOutputParser将模型的自由文本输出转换为可编程处理的XML数据。注意：它不会直接将模型的输出保持为原始XML字符串，而是会解析XML并转换成Python字典（或类似结构化的数据），目的是为了方便程序后续处理数据。

In [ ]:
from langchain_core.output_parsers import XMLOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_deepseek import ChatDeepSeek
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

# 创建 XML 输出解析器实例
parser = XMLOutputParser()

# 获取格式化指令（告诉模型如何以XML格式输出）
format_instructions = parser.get_format_instructions()

chat_prompt = ChatPromptTemplate.from_messages([
    ("system", f"你是一个AI助手，只能输出XML格式的结构化数据。{format_instructions}"),
    ("human", "请生成5个关于{topic}的内容，每个内容包含<name>和<description>两个字段")
])

prompt = chat_prompt.format_messages(topic="小米", format_instructions=format_instructions)
logger.info(prompt)

model = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=deepseek_api_key)

result = model.invoke(prompt)
logger.info(f"模型原始输出:\n{result.content}")

# 解析XML输出为结构化Python字典
response = parser.invoke(result)
logger.info(f"解析后的结构化结果:\n{response}")
logger.info(f"结果类型: {type(response)}")

### 4.5 PydanticOutputParser解析器

PydanticOutputParser是LangChain输出解析器体系中最常用、最强大的结构化解析器之一。它与JsonOutputParser类似，但功能更强——能直接基于Pydantic模型定义输出结构，并利用其类型校验与自动文档能力。对于结构更复杂、具有强类型约束的需求，PydanticOutputParser则是最佳选择。

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_deepseek import ChatDeepSeek
import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger()

from pydantic import BaseModel, Field, field_validator

class Product(BaseModel):
    name: str = Field(description="产品名称")
    category: str = Field(description="产品类别")
    description: str = Field(description="产品简介")

    @field_validator("description")
    def validate_description(cls, value):
        if len(value) < 10:
            raise ValueError('产品简介长度必须大于等于10')
        return value

# 创建Pydantic输出解析器，绑定Product模型
parser = PydanticOutputParser(pydantic_object=Product)

# 获取格式化指令，指导模型输出符合Product模型的JSON格式
format_instructions = parser.get_format_instructions()

prompt_template = ChatPromptTemplate.from_messages([
    ("system", "你是一个AI助手，你只能输出结构化的json数据\n{format_instructions}"),
    ("human", "请你输出标题为：{topic}的新闻内容")
])

prompt = prompt_template.format_messages(topic="小米", format_instructions=format_instructions)
logger.info(prompt)

model = ChatDeepSeek(
    model="deepseek-chat",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
    api_key=deepseek_api_key)

result = model.invoke(prompt)
logger.info(f"模型原始输出:\n{result.content}")

# 使用解析器将模型结果解析为Product对象
response = parser.invoke(result)
logger.info(f"解析后的结构化结果:\n{response}")
logger.info(f"结果类型: {type(response)}")

## 五、实验总结

1. 使用 StrOutputParser 将模型输出转为纯字符串，理解了最基础的解析方式
2. 通过 JsonOutputParser 配合提示词和 get_format_instructions 两种方式实现 JSON 结构化输出，对比了两种方式的灵活度
3. 使用 CommaSeparatedListOutputParser 将文本转为 Python 列表，掌握了列表类数据的提取方法
4. 通过 XMLOutputParser 将 XML 文本解析为 Python 字典，理解了 XML 格式在结构化数据中的应用
5. 使用 PydanticOutputParser 配合 BaseModel 实现强类型校验输出，体验了类型约束在数据质量保障上的价值